<a href="https://colab.research.google.com/github/nikitask14/neural-networks-pytorch-from-first-principles/blob/main/03_Autograd_and_gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch

**Problem 1**

Suppose:

$$ y = w^2 + 1 $$

and

$$ w=2 $$


w is a weight inside a neural network.

During training, we want to know: If I change \(w\) a tiny bit, what happens to \(y\)?

In [28]:
w = torch.tensor(2.0, requires_grad = True)

**Breakdown:**


Create a tensor called **w**, give it the value 2.0, and tell PyTorch to track calculations involving **w** so that gradients can be computed later.

The 2. is just PyTorch’s compact way of displaying 2.0.



Simply put, pyTorch is tracking operations involving w.



In [29]:
y = (w ** 2) + 1
print(w)
print(y)

tensor(2., requires_grad=True)
tensor(5., grad_fn=<AddBackward0>)


grad_fn=<AddBackward0>:
The last operation that produced this tensor was Addition.

PyTorch didn’t just store 5. It stored something conceptually like:

$$ w^2 → + 1 →y $$


That remembered relationship is what will later allow PyTorch to work backwards and calculate:

dw/dy


In [30]:
y.backward()

Starting from y, go backward through the operations you recorded and compute the derivative with respect to every tracked tensor that contributed to y.

More simply, asks PyTorch to travel backward through that computation

In [31]:
print(w.grad)

tensor(4.)


w.grad → gives us the derivative with respect to w


**Problem 2:**


Use:

$$ y = 2w^2 + 3 $$

with:

$$ w=2 $$

First, just calculate the two things yourself:

$$ y = ? $$

and

$$ \frac{dy}{dw} = ? $$

In [32]:
w = torch.tensor(2.0, requires_grad= True)
y = 2* (w ** 2) + 3
print(w)
print(y)

y.backward()
print(w.grad)


tensor(2., requires_grad=True)
tensor(11., grad_fn=<AddBackward0>)
tensor(8.)


So the idea of an Autograd is to tell pyTorch which value to track, perform the calculation, ask pyTorch to go backwards and read the resulting gradient.

Next, we should see how pyTorch uses Autograd to compute gradients.

In [33]:
w = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(2.0)
y = torch.tensor(4.0)

print(w)
print(x)
print(y)

y_hat = w*x
L = 0.5 * (y_hat - y) ** 2

print(y_hat)
print(L)
L.backward()
print(w.grad)

lr = 0.1
with torch.no_grad():
  w -= lr * w.grad


print(w)

tensor(1., requires_grad=True)
tensor(2.)
tensor(4.)
tensor(2., grad_fn=<MulBackward0>)
tensor(2., grad_fn=<MulBackward0>)
tensor(-4.)
tensor(1.4000, requires_grad=True)




*   print(y_hat)
*   print(L)


**Output:**


tensor(2., grad_fn=<MulBackward0>)


tensor(2., grad_fn=<MulBackward0>)

PyTorch is tracking them because they ultimately depend on the tracked weight w.


with torch.no_grad():

Read it as:

Change the value of w from - to -, but don't add “how we changed - into -” to the history PyTorch is recording.



**Problem 3:**

A short chain of operations.

Take:

$$ w = 2 $$ $$ z = w^2 $$ $$ y = 3z $$

Calculate:

$$ z = ? $$ $$ y = ? $$

and then:


dy/dw =?

In [34]:
w = torch.tensor(2.0, requires_grad = True)
z = w ** 2
y = 3 * z
y.backward()
print(w)
print(z)
print(y)
print(w.grad)


tensor(2., requires_grad=True)
tensor(4., grad_fn=<PowBackward0>)
tensor(12., grad_fn=<MulBackward0>)
tensor(12.)


Introducing more than one learnable parameter

In [41]:
w1 = torch.tensor(2.0, requires_grad = True)
b = torch.tensor(1.0, requires_grad = True)
x = torch.tensor(3.0)
print(w1)
print(b)
print(x)

y_hat = w1*x + b

y_hat.backward()

print(y_hat)
print(w1.grad)
print(w1.requires_grad)
print(b.grad)

tensor(2., requires_grad=True)
tensor(1., requires_grad=True)
tensor(3.)
tensor(7., grad_fn=<AddBackward0>)
tensor(3.)
True
tensor(1.)


In [43]:
w5 = torch.tensor(2.0, requires_grad = True)
y = w5 ** 2
print(w)
print(y)
y.backward()
print(w5.grad)

y = w5 ** 2
print(w)
print(y)
y.backward()
print(w5.grad)

tensor(2., requires_grad=True)
tensor(4., grad_fn=<PowBackward0>)
tensor(4.)
tensor(2., requires_grad=True)
tensor(4., grad_fn=<PowBackward0>)
tensor(8.)


It adds the new gradient to the existing value in w.grad.

So:

After the first backward pass:

$$ w_5.grad = 4 $$

If we compute

$$ y = w_5^2 $$

again and call .backward() again, PyTorch computes another gradient of 4 and adds it:

$$ w_5.grad = 4 + 4 = 8 $$

This behavior is called gradient accumulation.

So the key point is:

.backward() does not replace the old gradient by default. It accumulates onto it.

In [44]:
w.grad.zero_()

tensor(0.)

Clearing accumulated gradient.